# KYRBS 2023–2024: SAS → R analysis

This notebook reproduces the main steps in the supplied SAS program using R and the `survey` package.

**Flow:** import data → BMI classification → smartphone-use groups → covariates → missing-data exclusion → survey design → Table 1 (moonBook + Excel) → obesity prevalence (Table 2) → time-specific odds ratios.

The local SAS data files are expected in:
`C:/Users/picks/OneDrive/문서/Main/01_Others/논문/비만/Data`

## Cell 1 — Install packages (run once)

If these packages are already installed, skip this cell.

In [1]:
install.packages(c("haven", "dplyr", "tidyr", "survey", "broom", "moonBook", "openxlsx"))

Installing packages into 'C:/Users/picks/AppData/Local/R/win-library/4.6'
(as 'lib' is unspecified)



package 'haven' successfully unpacked and MD5 sums checked


Warning message:
"cannot remove prior installation of package 'haven'"
Warning message in file.copy(savedcopy, lib, recursive = TRUE):
"problem copying C:\Users\picks\AppData\Local\R\win-library\4.6\00LOCK\haven\libs\x64\haven.dll to C:\Users\picks\AppData\Local\R\win-library\4.6\haven\libs\x64\haven.dll: Permission denied"
Warning message:
"restored 'haven'"


package 'dplyr' successfully unpacked and MD5 sums checked


Warning message:
"cannot remove prior installation of package 'dplyr'"
Warning message in file.copy(savedcopy, lib, recursive = TRUE):
"problem copying C:\Users\picks\AppData\Local\R\win-library\4.6\00LOCK\dplyr\libs\x64\dplyr.dll to C:\Users\picks\AppData\Local\R\win-library\4.6\dplyr\libs\x64\dplyr.dll: Permission denied"
Warning message:
"restored 'dplyr'"


package 'tidyr' successfully unpacked and MD5 sums checked


Warning message:
"cannot remove prior installation of package 'tidyr'"
Warning message in file.copy(savedcopy, lib, recursive = TRUE):
"problem copying C:\Users\picks\AppData\Local\R\win-library\4.6\00LOCK\tidyr\libs\x64\tidyr.dll to C:\Users\picks\AppData\Local\R\win-library\4.6\tidyr\libs\x64\tidyr.dll: Permission denied"
Warning message:
"restored 'tidyr'"


package 'survey' successfully unpacked and MD5 sums checked


Warning message:
"cannot remove prior installation of package 'survey'"
Warning message in file.copy(savedcopy, lib, recursive = TRUE):
"problem copying C:\Users\picks\AppData\Local\R\win-library\4.6\00LOCK\survey\libs\x64\survey.dll to C:\Users\picks\AppData\Local\R\win-library\4.6\survey\libs\x64\survey.dll: Permission denied"
Warning message:
"restored 'survey'"


package 'broom' successfully unpacked and MD5 sums checked

The downloaded binary packages are in
	C:\Users\Public\Documents\ESTsoft\CreatorTemp\RtmpIBBX2d\downloaded_packages


## Cell 2 — Load packages

In [2]:
library(haven)
library(dplyr)
library(tidyr)
library(survey)
library(broom)
library(moonBook)
library(openxlsx)

Warning message:
"package 'haven' was built under R version 4.6.1"
Warning message:
"package 'dplyr' was built under R version 4.6.1"

Attaching package: 'dplyr'


The following objects are masked from 'package:stats':

    filter, lag


The following objects are masked from 'package:base':

    intersect, setdiff, setequal, union


Warning message:
"package 'survey' was built under R version 4.6.1"
Loading required package: grid

Loading required package: Matrix


Attaching package: 'Matrix'


The following objects are masked from 'package:tidyr':

    expand, pack, unpack


Loading required package: survival


Attaching package: 'survey'


The following object is masked from 'package:graphics':

    dotchart


Warning message:
"package 'broom' was built under R version 4.6.1"


## Cell 3 — Set the data directory and check files

In [3]:
DATA_DIR <- "C:/Users/picks/OneDrive/문서/Main/01_Others/논문/비만/Data"

FILE_2023 <- file.path(DATA_DIR, "kyrbs2023.sas7bdat")
FILE_2024 <- file.path(DATA_DIR, "kyrbs2024.sas7bdat")

cat("2023 exists:", file.exists(FILE_2023), "\n")
cat("2024 exists:", file.exists(FILE_2024), "\n")

if (!file.exists(FILE_2023) || !file.exists(FILE_2024)) {
  stop("SAS files were not found. Check DATA_DIR and filenames.")
}

2023 exists: TRUE 
2024 exists: TRUE 


## Cell 4 — Read 2023 and 2024 SAS datasets and combine them

This corresponds to the SAS `data ky; set a.kyrbs2023 a.kyrbs2024; run;` step.

In [4]:
# File paths

DATA_DIR <- "C:/Users/picks/OneDrive/문서/Main/01_Others/논문/비만/Data"

FILE_2023 <- file.path(DATA_DIR, "kyrbs2023.sas7bdat")
FILE_2024 <- file.path(DATA_DIR, "kyrbs2024.sas7bdat")

FILE_2023
FILE_2024

file.exists(FILE_2023)
file.exists(FILE_2024)

KY23 <- haven::read_sas(FILE_2023, encoding = "CP949")
KY24 <- haven::read_sas(FILE_2024, encoding = "CP949")

KY <- dplyr::bind_rows(KY23, KY24)

dim(KY)


[1] "C:/Users/picks/OneDrive/문서/Main/01_Others/논문/비만/Data/kyrbs2023.sas7bdat"

[1] "C:/Users/picks/OneDrive/문서/Main/01_Others/논문/비만/Data/kyrbs2024.sas7bdat"

[1] TRUE

[1] TRUE

[1] 107533    210

In [5]:
# Verify that the original SAS variable names are uppercase
head(names(KY), 30)
stopifnot("HT" %in% names(KY), "WT" %in% names(KY))


[1] "OBS"        "mod_d"      "YEAR"       "CITY"       "CTYPE"     
 [6] "CTYPE_SD"   "MH"         "SCHOOL"     "STYPE"      "STRATA"    
[11] "STRATA_NM"  "CLUSTER"    "GROUP"      "W"          "PR_HT"     
[16] "PR_BI"      "PR_HD"      "F_BR"       "F_FRUIT"    "F_SWD_A"   
[21] "F_FASTFOOD" "F_EDU"      "F_WAT"      "PA_TOT"     "PA_VIG_D"  
[26] "PA_MSC"     "PA_SWD_S"   "PA_SWD_N"   "PA_SWK_S"   "PA_SWK_N"

## Cell 5 — Calculate BMI

In [6]:
A2 <- KY %>%
  mutate(
    K = HT / 100,
    BMI = round(WT / (K * K), 9)
  )

## Cell 6 — Assign sex/age-specific BMI percentile cutoffs

This reproduces the cutoff values in the SAS code.

In [7]:
A2 <- A2 %>%
  mutate(
    PCT05 = case_when(
      SEX == 1 & AGE_M == 144 ~ 15.5,
      SEX == 1 & AGE_M == 145 ~ 15.6,
      SEX == 1 & AGE_M == 146 ~ 15.6,
      SEX == 1 & AGE_M >= 147 & AGE_M <= 227 ~ 17.0,
      SEX == 2 & AGE_M == 144 ~ 15.3,
      SEX == 2 & AGE_M == 145 ~ 15.3,
      SEX == 2 & AGE_M == 146 ~ 15.4,
      SEX == 2 & AGE_M >= 147 & AGE_M <= 227 ~ 16.5,
      TRUE ~ NA_real_
    ),
    PCT85 = case_when(
      SEX == 1 & AGE_M == 144 ~ 23.0,
      SEX == 1 & AGE_M == 145 ~ 23.0,
      SEX == 1 & AGE_M == 146 ~ 23.1,
      SEX == 1 & AGE_M >= 147 & AGE_M <= 227 ~ 24.5,
      SEX == 2 & AGE_M == 144 ~ 22.1,
      SEX == 2 & AGE_M == 145 ~ 22.2,
      SEX == 2 & AGE_M == 146 ~ 22.2,
      SEX == 2 & AGE_M >= 147 & AGE_M <= 227 ~ 23.5,
      TRUE ~ NA_real_
    ),
    PCT95 = case_when(
      SEX == 1 & AGE_M == 144 ~ 25.1,
      SEX == 1 & AGE_M == 145 ~ 25.1,
      SEX == 1 & AGE_M == 146 ~ 25.2,
      SEX == 1 & AGE_M >= 147 & AGE_M <= 227 ~ 26.5,
      SEX == 2 & AGE_M == 144 ~ 24.1,
      SEX == 2 & AGE_M == 145 ~ 24.2,
      SEX == 2 & AGE_M == 146 ~ 24.2,
      SEX == 2 & AGE_M >= 147 & AGE_M <= 227 ~ 25.5,
      TRUE ~ NA_real_
    )
  )

## Cell 7 — Create BMI groups (`g_bmi`)

In [8]:
A3 <- A2 %>%
  mutate(
    G_BMI = case_when(
      !is.na(BMI) & !is.na(PCT05) & !is.na(PCT85) & !is.na(PCT95) & BMI >= PCT95 ~ 4,
      !is.na(BMI) & !is.na(PCT05) & !is.na(PCT85) & !is.na(PCT95) & BMI >= PCT85 & BMI < PCT95 ~ 3,
      !is.na(BMI) & !is.na(PCT05) & !is.na(PCT85) & !is.na(PCT95) & BMI >= PCT05 & BMI < PCT85 ~ 2,
      !is.na(BMI) & !is.na(PCT05) & !is.na(PCT85) & !is.na(PCT95) & BMI < PCT05 ~ 1,
      TRUE ~ NA_real_
    )
  )

## Cell 8 — Calculate average daily smartphone use

In [9]:
A4 <- A3 %>%
  mutate(
    SP_WD_HR = INT_SPWD_TM / 60,
    SP_WK_HR = INT_SPWK_TM / 60,
    SP_AVG = (SP_WD_HR * 5 + SP_WK_HR * 2) / 7
  )

## Cell 9 — Create smartphone-use groups (`time`)

The SAS code uses chained comparisons such as `2 <= sp_avg < 4`. In R, these must be written with `&`.

In [10]:
A4 <- A4 %>%
  mutate(
    TIME = case_when(
      SP_AVG < 2 ~ 1,
      SP_AVG >= 2 & SP_AVG < 4 ~ 2,
      SP_AVG >= 4 & SP_AVG < 6 ~ 3,
      SP_AVG >= 6 ~ 4,
      TRUE ~ NA_real_
    )
  )

## Cell 10 — Create region, smoking, education, economic status, stress, depression, drinking, and age group

The SAS program maps `군지역` to region 2 and `대도시/중소도시` to region 1.

In [11]:
A4 <- A4 %>%
  mutate(
    REGION = case_when(
      as.character(CTYPE) == "군지역" ~ 2,
      as.character(CTYPE) %in% c("대도시", "중소도시") ~ 1,
      TRUE ~ NA_real_
    ),

    SMOKING1 = if_else(TC_DAYS %in% c(1, 9999) | is.na(TC_DAYS), 0, 1),
    SMOKING2 = if_else(TC_EC_MN %in% c(1, 9999) | is.na(TC_EC_MN), 0, 1),
    SMOKING3 = if_else(TC_HTP_MN %in% c(1, 9999) | is.na(TC_HTP_MN), 0, 1),
    SMOKING = if_else(SMOKING1 == 1 | SMOKING2 == 1 | SMOKING3 == 1, 1, 0),

    EDU = case_when(
      E_S_RCRD %in% c(1, 2) ~ 1,
      E_S_RCRD == 3 ~ 2,
      E_S_RCRD %in% c(4, 5) ~ 3,
      TRUE ~ NA_real_
    ),

    ECO = case_when(
      E_SES %in% c(1, 2) ~ 1,
      E_SES == 3 ~ 2,
      E_SES %in% c(4, 5) ~ 3,
      TRUE ~ NA_real_
    ),

    STRESS = if_else(M_STR %in% c(1, 2), 1, 0),
    DEPRESS = if_else(M_SAD == 2, 1, 0),
    DRINKING = if_else(AC_DAYS %in% c(1, 9999) | is.na(AC_DAYS), 0, 1),

    AGE_G = case_when(
      AGE %in% c(12, 13, 14) ~ 1,
      AGE %in% c(15, 16, 17, 18) ~ 2,
      TRUE ~ NA_real_
    )
  ) %>%
  filter(YEAR %in% c(2023, 2024))

## Cell 11 — Keep the same analysis variables as the SAS program

In [12]:
A4 <- A4 %>%
  select(
    YEAR, AGE_G, SEX, REGION, G_BMI, TIME,
    SMOKING, DRINKING, EDU, ECO, STRESS, DEPRESS,
    W, CLUSTER, STRATA
  )

## Cell 12 — Check frequencies before deleting missing observations

This corresponds to the first `PROC FREQ` in the SAS program.

In [13]:
freq_vars <- c("YEAR", "SEX", "AGE_G", "REGION", "G_BMI", "EDU", "ECO",
               "SMOKING", "DRINKING", "TIME", "STRESS", "DEPRESS")

for (v in freq_vars) {
  cat("\n====================", v, "====================\n")
  print(table(A4[[v]], useNA = "ifany"))
}


==================== YEAR ====================

 2023  2024 
52880 54653 

==================== SEX ====================

    1     2 
54859 52674 

==================== AGE_G ====================

    1     2  <NA> 
45619 61787   127 

==================== REGION ====================

    1     2 
99520  8013 

==================== G_BMI ====================

    1     2     3     4  <NA> 
 7102 74988 10028 12522  2893 

==================== EDU ====================

    1     2     3  <NA> 
40879 31384 35262     8 

==================== ECO ====================

    1     2     3  <NA> 
45553 49412 12558    10 

==================== SMOKING ====================

     0      1 
102406   5127 

==================== DRINKING ====================

    0     1 
96504 11029 

==================== TIME ====================

    1     2     3     4  <NA> 
 7598 32989 31714 31129  4103 

==================== STRESS ====================

    0     1 
64792 42741 

==================== DEPRESS

## Cell 13 — Exclude records with missing age group, BMI group, education, or economic status

This matches the SAS deletion rule.

In [14]:
A5 <- A4 %>%
  filter(
    !is.na(AGE_G),
    !is.na(G_BMI),
    !is.na(EDU),
    !is.na(ECO)
  )

## Cell 14 — Create the binary obesity outcome

In [15]:
A5 <- A5 %>%
  mutate(
    OBESE = case_when(
      G_BMI %in% c(1, 2, 3) ~ 0,
      G_BMI == 4 ~ 1,
      TRUE ~ NA_real_
    )
  )

## Cell 15 — Check the final analytic sample

In [16]:
cat("Final N:", nrow(A5), "\n")

for (v in c(freq_vars, "OBESE")) {
  cat("\n====================", v, "====================\n")
  print(table(A5[[v]], useNA = "ifany"))
}

Final N: 104630 

==================== YEAR ====================

 2023  2024 
51462 53168 

==================== SEX ====================

    1     2 
53469 51161 

==================== AGE_G ====================

    1     2 
44499 60131 

==================== REGION ====================

    1     2 
96912  7718 

==================== G_BMI ====================

    1     2     3     4 
 7102 74981 10027 12520 

==================== EDU ====================

    1     2     3 
40050 30681 33899 

==================== ECO ====================

    1     2     3 
44496 48177 11957 

==================== SMOKING ====================

    0     1 
99861  4769 

==================== DRINKING ====================

    0     1 
94107 10523 

==================== TIME ====================

    1     2     3     4  <NA> 
 7461 32421 31025 30013  3710 

==================== STRESS ====================

    0     1 
63335 41295 

==================== DEPRESS ====================

    0     1 

## Cell 16 — Define the complex survey design

This corresponds to SAS `strata strata; cluster cluster; weight w;`.

In [17]:
options(survey.lonely.psu = "adjust")

DESIGN <- svydesign(
  ids = ~CLUSTER,
  STRATA = ~STRATA,
  weights = ~W,
  data = A5,
  nest = TRUE
)

DESIGN

1 - level Cluster Sampling design (with replacement)
With (800) clusters.
svydesign(ids = ~CLUSTER, STRATA = ~STRATA, weights = ~W, data = A5, 
    nest = TRUE)

## Cell 17 — Table 1: create the table with `moonBook`

`moonBook::mytable()` is used to create a quick descriptive Table 1 by smartphone-use group.

Important: `mytable()` is a descriptive-table function and does **not** apply the KYRBS complex survey design. Therefore, the Excel table below uses the actual analytic sample counts and within-group percentages, matching the manuscript Table 1 format. The `survey` design remains the correct tool for weighted prevalence and regression analyses.

In [18]:
# Convert analysis variables to labeled factors for a readable moonBook table
A5_labeled <- A5 %>%
  mutate(
    TIME = factor(TIME, levels = 1:4, labels = c("<2h", "2h–4h", "4h–6h", ">6h")),
    SEX = factor(SEX, levels = c(1, 2), labels = c("Male", "Female")),
    AGE_G = factor(AGE_G, levels = c(1, 2), labels = c("7th–9th grade", "10th–12th grade")),
    REGION = factor(REGION, levels = c(1, 2), labels = c("Urban", "Rural")),
    G_BMI = factor(G_BMI, levels = c(1, 2, 3, 4),
                   labels = c("Underweight", "Normal", "Overweight", "Obese")),
    EDU = factor(EDU, levels = c(1, 2, 3), labels = c("High", "Middle", "Low")),
    ECO = factor(ECO, levels = c(1, 2, 3), labels = c("High", "Middle", "Low")),
    STRESS = factor(STRESS, levels = c(0, 1), labels = c("Low", "High")),
    DEPRESS = factor(DEPRESS, levels = c(0, 1), labels = c("Low", "High")),
    SMOKING = factor(SMOKING, levels = c(0, 1), labels = c("Non-smoker", "Smoker")),
    DRINKING = factor(DRINKING, levels = c(0, 1), labels = c("Non-drinker", "Drinker"))
  )

# moonBook descriptive table
moonbook_table1 <- mytable(
  TIME ~ SEX + AGE_G + REGION + G_BMI + EDU + ECO + STRESS + DEPRESS +
    SMOKING + DRINKING,
  data = A5_labeled,
  show.total = TRUE,
  show.all = FALSE
)

moonbook_table1


==================== SEX ====================
      category  percent   ci_low  ci_high
1 factor(SEX)1 51.56042 50.04959 53.07125
2 factor(SEX)2 48.43958 46.92875 49.95041

==================== AGE_G ====================
        category  percent   ci_low  ci_high
1 factor(AGE_G)1 40.31146 38.15669 42.46623
2 factor(AGE_G)2 59.68854 57.53377 61.84331

==================== REGION ====================
         category   percent    ci_low   ci_high
1 factor(REGION)1 94.471064 93.368285 95.573842
2 factor(REGION)2  5.528936  4.426158  6.631715

==================== G_BMI ====================
        category   percent    ci_low   ci_high
1 factor(G_BMI)1  6.776858  6.526990  7.026727
2 factor(G_BMI)2 72.068243 71.660715 72.475771
3 factor(G_BMI)3  9.512949  9.285432  9.740466
4 factor(G_BMI)4 11.641950 11.301319 11.982581

==================== EDU ====================
      category  percent   ci_low  ci_high
1 factor(EDU)1 38.37635 37.70820 39.04449
2 factor(EDU)2 29.30904 28.95201 29.6

## Cell 18 — Table 1: make the manuscript-style Excel table

The manuscript Table 1 is arranged as:

**Characteristics | Total | <2h | 2h–4h | 4h–6h | >6h**

Values are shown as **n (%)**, where the percentage is calculated within the total sample or within each screen-time group.

In [19]:
TABLE1_VARS <- c(
  SEX = "Sex",
  AGE_G = "Grade",
  REGION = "Region of residence",
  G_BMI = "BMI*",
  EDU = "Academic achievement",
  ECO = "Economic level",
  STRESS = "Stress",
  DEPRESS = "Depression",
  SMOKING = "Smoking status",
  DRINKING = "Alcohol consumption"
)

TIME_LEVELS <- c("<2h", "2h–4h", "4h–6h", ">6h")

make_table1 <- function(data, variables, time_var = "TIME") {
  total_n <- nrow(data)
  result <- list()
  k <- 1

  # Overall row
  overall_row <- data.frame(
    Characteristics = "Overall",
    Total = sprintf("%d", total_n),
    `<2h` = sprintf("%d (%.2f)", sum(data[[time_var]] == "<2h", na.rm = TRUE),
                    mean(data[[time_var]] == "<2h", na.rm = TRUE) * 100),
    `2h–4h` = sprintf("%d (%.2f)", sum(data[[time_var]] == "2h–4h", na.rm = TRUE),
                      mean(data[[time_var]] == "2h–4h", na.rm = TRUE) * 100),
    `4h–6h` = sprintf("%d (%.2f)", sum(data[[time_var]] == "4h–6h", na.rm = TRUE),
                      mean(data[[time_var]] == "4h–6h", na.rm = TRUE) * 100),
    `>6h` = sprintf("%d (%.2f)", sum(data[[time_var]] == ">6h", na.rm = TRUE),
                    mean(data[[time_var]] == ">6h", na.rm = TRUE) * 100),
    check.names = FALSE,
    stringsAsFactors = FALSE
  )
  result[[k]] <- overall_row
  k <- k + 1

  for (v in names(variables)) {
    # Section/header row
    result[[k]] <- data.frame(
      Characteristics = variables[[v]],
      Total = "",
      `<2h` = "",
      `2h–4h` = "",
      `4h–6h` = "",
      `>6h` = "",
      check.names = FALSE,
      stringsAsFactors = FALSE
    )
    k <- k + 1

    levs <- levels(factor(data[[v]]))

    for (lev in levs) {
      total_count <- sum(!is.na(data[[v]]) & data[[v]] == lev)
      total_pct <- ifelse(total_n > 0, total_count / total_n * 100, NA_real_)

      row <- data.frame(
        Characteristics = paste0("  ", lev),
        Total = sprintf("%d (%.2f)", total_count, total_pct),
        `<2h` = "",
        `2h–4h` = "",
        `4h–6h` = "",
        `>6h` = "",
        check.names = FALSE,
        stringsAsFactors = FALSE
      )

      for (tt in TIME_LEVELS) {
        d <- data[data[[time_var]] == tt & !is.na(data[[v]]), , drop = FALSE]
        n_group <- nrow(d)
        n_level <- sum(d[[v]] == lev)
        pct_group <- ifelse(n_group > 0, n_level / n_group * 100, NA_real_)
        row[[tt]] <- sprintf("%d (%.2f)", n_level, pct_group)
      }

      result[[k]] <- row
      k <- k + 1
    }
  }

  bind_rows(result)
}

Table1 <- make_table1(A5_labeled, TABLE1_VARS)

Table1



########################################
TIME = 1 
########################################

--- SEX ---
      category  percent   ci_low  ci_high
1 factor(SEX)1 62.55724 60.40707 64.70741
2 factor(SEX)2 37.44276 35.29259 39.59293

--- AGE_G ---
        category  percent   ci_low  ci_high
1 factor(AGE_G)1 56.00808 53.00862 59.00755
2 factor(AGE_G)2 43.99192 40.99245 46.99138

--- REGION ---
         category   percent    ci_low   ci_high
1 factor(REGION)1 96.009185 94.912516 97.105855
2 factor(REGION)2  3.990815  2.894145  5.087484

--- G_BMI ---
        category   percent    ci_low   ci_high
1 factor(G_BMI)1 10.344634  9.524373 11.164895
2 factor(G_BMI)2 72.074037 70.858921 73.289154
3 factor(G_BMI)3  8.873367  8.158699  9.588036
4 factor(G_BMI)4  8.707962  7.942049  9.473875

--- EDU ---
      category  percent   ci_low  ci_high
1 factor(EDU)1 60.45880 59.06585 61.85175
2 factor(EDU)2 24.48943 23.37715 25.60172
3 factor(EDU)3 15.05177 14.12171 15.98182

--- ECO ---
      category  

## Cell 19 — Export Table 1 to Excel

This saves the manuscript-style Table 1 as an `.xlsx` file. The Excel sheet contains the same **n (%)** format shown in the paper.

In [ ]:
TABLE1_XLSX <- file.path(DATA_DIR, "Table1_screen_time.xlsx")

wb <- createWorkbook()
addWorksheet(wb, "Table 1")

writeData(wb, "Table 1", Table1, startRow = 1, startCol = 1, rowNames = FALSE)

# Basic formatting for a manuscript-style table
header_style <- createStyle(textDecoration = "bold", halign = "center", border = "Bottom")
section_style <- createStyle(textDecoration = "bold")

addStyle(wb, "Table 1", header_style,
         rows = 1, cols = 1:ncol(Table1), gridExpand = TRUE)

# Bold section rows (rows where only Characteristics is filled)
section_rows <- which(Table1$Total == "" & Table1$Characteristics != "Overall") + 1
if (length(section_rows) > 0) {
  addStyle(wb, "Table 1", section_style,
           rows = section_rows, cols = 1, gridExpand = FALSE)
}

setColWidths(wb, "Table 1", cols = 1, widths = 30)
setColWidths(wb, "Table 1", cols = 2:ncol(Table1), widths = 18)
freezePane(wb, "Table 1", firstRow = TRUE)

saveWorkbook(wb, TABLE1_XLSX, overwrite = TRUE)

cat("Table 1 Excel file saved to:", TABLE1_XLSX, "\n")

## Cell 20 — Table 2 helper: obesity prevalence

The SAS program creates `t1obese`, `t2obese`, etc. only for the corresponding smartphone-use group. In R, it is simpler and equivalent to subset the survey design by `time` and estimate the mean of the binary `obese` variable.

In [20]:
survey_prevalence <- function(design_obj) {
  est <- svymean(~OBESE, design_obj, na.rm = TRUE)
  ci <- confint(est)
  data.frame(
    prevalence = as.numeric(coef(est)) * 100,
    ci_low = ci[1] * 100,
    ci_high = ci[2] * 100
  )
}

prevalence_total <- bind_rows(lapply(1:4, function(t) {
  d <- subset(DESIGN, TIME == t)
  cbind(TIME = t, survey_prevalence(d))
}))

prevalence_total

TIME,prevalence,ci_low,ci_high
<int>,<dbl>,<dbl>,<dbl>
1,8.707962,7.942049,9.473875
2,10.448739,9.990898,10.906580
3,11.493028,11.048419,11.937638
4,13.902378,13.375268,14.429488


## Cell 21 — Table 2: obesity prevalence by risk-factor category and smartphone-use group

This replaces the very large block of `t1s1`, `t2s1`, `t1a1`, etc. variables in the SAS program.

In [21]:
RISK_FACTORS <- list(
  SEX = c(1, 2),
  AGE_G = c(1, 2),
  REGION = c(1, 2),
  EDU = c(1, 2, 3),
  ECO = c(1, 2, 3),
  DEPRESS = c(0, 1),
  STRESS = c(0, 1),
  DRINKING = c(0, 1),
  SMOKING = c(0, 1)
)

prevalence_by_risk <- function(design_obj, variable, level) {
  d <- subset(design_obj, get(variable) == level)
  if (nrow(d$variables) == 0) {
    return(data.frame(prevalence = NA_real_, ci_low = NA_real_, ci_high = NA_real_))
  }
  survey_prevalence(d)
}

prevalence_table2 <- bind_rows(
  lapply(names(RISK_FACTORS), function(v) {
    bind_rows(lapply(1:4, function(t) {
      bind_rows(lapply(RISK_FACTORS[[v]], function(level) {
        d <- subset(DESIGN, TIME == t)
        d_level <- subset(d, get(v) == level)
        if (nrow(d_level$variables) == 0) {
          data.frame(variable = v, level = level, TIME = t,
                     prevalence = NA_real_, ci_low = NA_real_, ci_high = NA_real_)
        } else {
          x <- survey_prevalence(d_level)
          data.frame(variable = v, level = level, TIME = t,
                     prevalence = x$prevalence, ci_low = x$ci_low, ci_high = x$ci_high)
        }
      }))
    }))
  })
)

prevalence_table2

variable,level,TIME,prevalence,ci_low,ci_high
<chr>,<dbl>,<int>,<dbl>,<dbl>,<dbl>
SEX,1,1,11.413149,10.397723,12.428576
SEX,2,1,4.188288,3.361880,5.014697
SEX,1,2,13.387477,12.754768,14.020186
SEX,2,2,6.676256,6.195905,7.156606
SEX,1,3,14.618516,13.934543,15.302488
SEX,2,3,8.500703,8.008471,8.992936
SEX,1,4,17.627666,16.826624,18.428708
SEX,2,4,10.974033,10.389931,11.558134
AGE_G,1,1,6.096582,5.308723,6.884442


## Cell 22 — Function for time-specific survey logistic regression

SAS uses `PROC SURVEYLOGISTIC` with `obese (event='1')` and one risk factor at a time.
The function below uses `svyglm(..., family = quasibinomial())` with the same survey strata, clusters, and weights.

In [25]:
fit_svy_or <- function(data, variable, reference) {
  data <- data %>%
    filter(!is.na(OBESE), !is.na(.data[[variable]]))

  data[[variable]] <- factor(data[[variable]])
  data[[variable]] <- relevel(data[[variable]], ref = as.character(reference))

  d <- svydesign(
    ids = ~CLUSTER,
    STRATA = ~STRATA,
    weights = ~W,
    data = data,
    nest = TRUE
  )

  f <- as.formula(paste("OBESE ~", variable))

  model <- svyglm(
    f,
    design = d,
    family = quasibinomial()
  )

  est <- coef(model)
  se <- sqrt(diag(vcov(model)))
  df <- degf(d)
  crit <- qt(0.975, df = df)

  result <- data.frame(
    term = names(est),
    OR = exp(est),
    CI_low = exp(est - crit * se),
    CI_high = exp(est + crit * se),
    p_value = 2 * pt(abs(est / se), df = df, lower.tail = FALSE),
    row.names = NULL
  )

  result %>%
    filter(term != "(Intercept)") %>%
    mutate(
      TIME = unique(data$TIME),
      risk_factor = variable,
      .before = 1
    )
}

## Cell 23 — Run the nine unadjusted OR analyses for TIME = 1, 2, 3, 4

Reference categories exactly follow the SAS code:
- sex: 2
- age_g: 1
- region: 1
- edu: 1
- eco: 1
- depress/stress/smoking/drinking: 0

In [26]:
reference_values <- c(
  SEX = 2,
  AGE_G = 1,
  REGION = 1,
  EDU = 1,
  ECO = 1,
  DEPRESS = 0,
  STRESS = 0,
  SMOKING = 0,
  DRINKING = 0
)

or_table <- bind_rows(lapply(1:4, function(t) {
  dat <- A5 %>% filter(TIME == t)

  bind_rows(lapply(names(reference_values), function(v) {
    fit_svy_or(dat, v, reference_values[[v]])
  }))
}))

or_table

TIME,risk_factor,term,OR,CI_low,CI_high,p_value
<dbl>,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>
1,SEX,SEX1,2.9472584,2.3654995,3.672092,6.616172e-21
1,AGE_G,AGE_G2,2.1068506,1.7587065,2.523911,2.087330e-15
1,REGION,REGION2,1.1480009,0.7785583,1.692752,4.855829e-01
1,EDU,EDU2,1.5880033,1.2951073,1.947140,9.697858e-06
1,EDU,EDU3,2.1927071,1.7495863,2.748058,1.727008e-11
1,ECO,ECO2,1.1354191,0.9435478,1.366308,1.784410e-01
1,ECO,ECO3,1.7968317,1.3569026,2.379393,4.627986e-05
1,DEPRESS,DEPRESS1,1.0143393,0.8247637,1.247490,8.925829e-01
1,STRESS,STRESS1,1.1191562,0.9399105,1.332585,2.058738e-01


## Cell 24 — Format OR results for a paper/table

This creates `OR (95% CI)` text and keeps the p-value separately.

In [27]:
or_table_formatted <- or_table %>%
  mutate(
    OR_CI = sprintf("%.2f (%.2f–%.2f)", OR, CI_low, CI_high),
    p_value = ifelse(p_value < 0.001, "<0.001", sprintf("%.3f", p_value))
  ) %>%
  select(TIME, risk_factor, term, OR_CI, p_value)

or_table_formatted

TIME,risk_factor,term,OR_CI,p_value
<dbl>,<chr>,<chr>,<chr>,<chr>
1,SEX,SEX1,2.95 (2.37–3.67),<0.001
1,AGE_G,AGE_G2,2.11 (1.76–2.52),<0.001
1,REGION,REGION2,1.15 (0.78–1.69),0.486
1,EDU,EDU2,1.59 (1.30–1.95),<0.001
1,EDU,EDU3,2.19 (1.75–2.75),<0.001
1,ECO,ECO2,1.14 (0.94–1.37),0.178
1,ECO,ECO3,1.80 (1.36–2.38),<0.001
1,DEPRESS,DEPRESS1,1.01 (0.82–1.25),0.893
1,STRESS,STRESS1,1.12 (0.94–1.33),0.206


## Cell 25 — Optional: save the main results as CSV files

In [28]:
write.csv(prevalence_total, file.path(DATA_DIR, "Table2_total_obesity_prevalence.csv"), row.names = FALSE)
write.csv(prevalence_table2, file.path(DATA_DIR, "Table2_risk_factor_obesity_prevalence.csv"), row.names = FALSE)
write.csv(or_table_formatted, file.path(DATA_DIR, "OR_time_specific.csv"), row.names = FALSE)

cat("Results saved to:", DATA_DIR, "\n")

Results saved to: C:/Users/picks/OneDrive/문서/Main/01_Others/논문/비만/Data 


## Cell 26 — Final checks

Use these checks before comparing the R results with the original SAS output.

In [29]:
cat("Analytic N:", nrow(A5), "\n")
cat("PSUs:", length(unique(A5$CLUSTER)), "\n")
cat("Strata:", length(unique(A5$STRATA)), "\n")
cat("\nObesity distribution:\n")
print(table(A5$OBESE, useNA = "ifany"))
cat("\nSmartphone-use groups:\n")
print(table(A5$TIME, useNA = "ifany"))
cat("\nOR results:\n")
print(or_table_formatted)

Analytic N: 104630 
PSUs: 800 
Strata: 216 

Obesity distribution:

    0     1 
92110 12520 

Smartphone-use groups:

    1     2     3     4  <NA> 
 7461 32421 31025 30013  3710 

OR results:
   TIME risk_factor      term            OR_CI p_value
1     1         SEX      SEX1 2.95 (2.37–3.67)  <0.001
2     1       AGE_G    AGE_G2 2.11 (1.76–2.52)  <0.001
3     1      REGION   REGION2 1.15 (0.78–1.69)   0.486
4     1         EDU      EDU2 1.59 (1.30–1.95)  <0.001
5     1         EDU      EDU3 2.19 (1.75–2.75)  <0.001
6     1         ECO      ECO2 1.14 (0.94–1.37)   0.178
7     1         ECO      ECO3 1.80 (1.36–2.38)  <0.001
8     1     DEPRESS  DEPRESS1 1.01 (0.82–1.25)   0.893
9     1      STRESS   STRESS1 1.12 (0.94–1.33)   0.206
10    1     SMOKING  SMOKING1 0.99 (0.39–2.47)   0.978
11    1    DRINKING DRINKING1 1.69 (1.17–2.44)   0.006
12    2         SEX      SEX1 2.16 (1.97–2.37)  <0.001
13    2       AGE_G    AGE_G2 1.74 (1.59–1.91)  <0.001
14    2      REGION   REGION2 1.25 (